## Bucketing

Bucketing - The One Spark Optimization You're Not Doing

**1. Introduction to Bucketing**

*   **Bucketing is a technique to divide datasets into more manageable chunks**.
*   This division is **based on the hash value of the data**.
*   The primary goal of bucketing is to **improve the performance of Spark queries**.
*   It specifically enhances operations like **filter, aggregation (group by), or join**.

**2. Filter Operation and Bucketing**

*   **Problem with Storing Data as is for Filtering:** Scanning all records in a large, undivided dataset is inefficient.
*   **Problem with Partitioning by High Cardinality Columns:** Partitioning by a column with many unique values (high cardinality), like `product ID`, leads to a **large number of small partitions (small file problem)**, which is also inefficient.
*   **Bucketing Solution for Filtering:**
    *   Rows are divided into a **fixed number of buckets**.
    *   The bucket for each row is determined by the **hash of the bucketing key (e.g., `product ID`) modulo the number of buckets**.
    *   **Example:** Bucketing by `product ID` into 4 buckets. A `product ID` of 22 would go to bucket 2 (hash(22) mod 4 = 2).
    *   When filtering for a specific value (e.g., `product ID` = 22), Spark **only needs to examine the corresponding bucket**.
    *   This **reduces the search space** and makes filter queries much more efficient.

**3. Join Operation and Bucketing**

*   **Problem with Joining Non-Bucketed DataFrames:** Involves a costly **shuffle, sort, and merge** operation to bring data with the same join keys to the same partition. The **shuffle** is particularly expensive as it redistributes data across the network.
*   **Problem with Partitioning for Joins on High Cardinality Columns:** Similar to filtering, partitioning both dataframes on a high cardinality join key can lead to the small file problem.
*   **Bucketing Solution for Joins:**
    *   **Both dataframes should be bucketed on the join key** (e.g., `product ID`) with the **same number of buckets**.
    *   The logic for assigning rows to buckets is the same: hash of the join key modulo the number of buckets.
    *   When joining bucketed dataframes, **buckets with the same hash value (for the join key) reside on the same executor**.
    *   The **shuffle step is avoided** because data with matching join keys is already co-located in the same buckets on the same machine.
    *   Only the **sort and merge steps are required** for the join operation.
    *   **Repeated joins on the same bucketed dataframes benefit the most** as the shuffle is avoided in subsequent joins.
*   **Scenarios for Joins on Bucketed DataFrames:**
    *   **Both bucketed with the same number of buckets on the join key:** No shuffle. This is the optimal scenario.
    *   **Both bucketed with a different number of buckets on the join key:** One of the dataframes will likely be shuffled to match the number of buckets of the other.
    *   **Both bucketed on a column other than the join key:** Bucketing provides no benefit for the join, and a **full shuffle will occur**.

**4. Group By Operation and Bucketing**

*   **Problem with Group By on Non-Bucketed DataFrames:** Typically involves a **shuffle** to bring all rows with the same group by key to the same partition for aggregation.
*   **Bucketing Solution for Group By:**
    *   If a dataframe is **bucketed on the group by key**, the data with the same key is already in the same buckets on the same executors.
    *   The **shuffle step is eliminated or significantly reduced**.
    *   Only the **aggregation step** (e.g., sum) needs to be performed locally within each bucket and then potentially combined for a final result.
    *   **Example:** Calculating total sales by `product ID` on a dataframe bucketed by `product ID` avoids the initial shuffle required to group all sales for the same `product ID` together.

**5. Determining the Optimal Number of Buckets**

*   The number of buckets should be chosen based on the **size of the dataset**.
*   **Optimal bucket size:** Generally between **128 to 200 MB**.
*   **Formula for the number of buckets:** `Size of the data set / Optimal bucket size`.
*   **Example:** For a 1 GB (1000 MB) dataset, with an optimal bucket size of 200 MB, the optimal number of buckets would be 5 (1000 / 200 = 5).

**6. Estimating Data Set Size**

*   A formula to estimate the size of a DataFrame is provided: `n * v * w / 1024^2`.
    *   `n`: Number of records.
    *   `v`: Number of variables (columns).
    *   `w`: Approximate average width (in bytes) of a variable/column.
        *   Small integer: width = 1.
        *   Medium to large integer: width = 2 to 4.
        *   Floats and strings: Calculated based on their size.
*   Calculating `n` requires at least one scan of the dataset.
*   Bucketing is often a **one-time operation** after which the performance benefits can be realized for multiple subsequent queries.

**7. Code Demonstration Highlights**

*   **Joining Non-Bucketed DataFrames:** The physical plan shows **two "Exchange" operations** (shuffles) before the sort merge join.
*   **Bucketing DataFrames:** Achieved using `dataFrame.write.bucketBy("columnName", numBuckets).saveAsTable("tableName")`. Bucketed data is stored in the Spark warehouse.
*   **Joining Bucketed DataFrames:** When joining dataframes bucketed on the join key with the same number of buckets, the physical plan **does not show the "Exchange" (shuffle) operation**. This indicates that the shuffle is avoided.
*   **Aggregation on Non-Bucketed DataFrames:** The physical plan shows an **"Exchange hashpartitioning" operation** (shuffle) before the global aggregation.
*   **Aggregation on Bucketed DataFrames:** When performing a group by on a dataframe bucketed by the group by key, the **"Exchange" (shuffle) operation is absent** in the physical plan.
*   **Bucket Pruning for Filters:** When filtering on the bucketing key, Spark is intelligent enough to **only read the relevant bucket(s)**, significantly reducing I/O and processing time. The physical plan shows that only a subset of the total buckets is scanned.

**8. Conclusion**

*   Bucketing is a powerful optimization technique in Spark.
*   It can significantly **speed up filter, join, and group by operations** by:
    *   **Avoiding costly shuffle operations.**
    *   **Reducing the amount of data that needs to be scanned or processed.**
*   Properly choosing the bucketing key (relevant to common filter, join, and group by operations) and the number of buckets is crucial to maximize the benefits of bucketing.

# Questions


1.  **Which of the following is the primary basis for how bucketing divides a Spark DataFrame?**
    *   The data type of a specified column.
    *   The range of values within a specified column.
    *   The **hash value** of the data in a specified bucketing column.
    *   The physical order of records as they are read into the DataFrame.

2.  **Unlike partitioning, bucketing in Spark guarantees:**
    *   Data with the same value in the bucketing column will always reside in the same partition.
    *   The number of output files will always match the number of partitions.
    *   A **fixed number of buckets** regardless of the cardinality of the bucketing column.
    *   Smaller file sizes compared to partitioning on a high-cardinality column.

3.  **For a filter operation on a high-cardinality column, why might bucketing be preferred over partitioning by the same column?**
    *   Bucketing inherently creates fewer output files.
    *   Bucketing automatically sorts data within each bucket, improving filter speed.
    *   Partitioning on a high-cardinality column can lead to the **small file problem**, whereas bucketing controls the number of files.
    *   Bucketing preserves the original partitioning scheme of the data.

4.  **Consider two Spark DataFrames, `DF1` bucketed into 10 buckets by column `A`, and `DF2` also bucketed into 10 buckets by column `A`. When joining `DF1` and `DF2` on column `A`, what is the most significant performance advantage you would expect compared to joining non-bucketed versions of the same DataFrames?**
    *   Reduced disk I/O during the join.
    *   Elimination or significant reduction of the **shuffle operation**.
    *   Automatic optimization of the join algorithm to broadcast join.
    *   Data skew is automatically handled more effectively.

5.  **If you have two DataFrames, one bucketed into `X` buckets by `product_id` and another bucketed into `Y` buckets by `product_id` where `X` is not equal to `Y`, and you perform a join on `product_id`, what is the likely outcome regarding the shuffle operation?**
    *   Both DataFrames will avoid shuffling.
    *   The larger DataFrame (in terms of rows) will be shuffled to match the number of buckets of the smaller one.
    *   **One of the DataFrames will likely be shuffled** to have the same number of buckets as the other to facilitate the bucketed join.
    *   The join will automatically revert to a broadcast join strategy.

6.  **In which of the following scenarios would bucketing on a specific column *not* provide a significant performance benefit for a join operation between two DataFrames?**
    *   Both DataFrames are bucketed on the join key with the same number of buckets.
    *   Both DataFrames are bucketed on the join key, but with a different number of buckets.
    *   **Both DataFrames are bucketed on a column that is *different* from the join key**.
    *   One DataFrame is bucketed on the join key, and the other is not bucketed.

7.  **When performing a `groupBy` operation on a DataFrame that has been bucketed by the same group-by key, what is the primary performance gain observed in the Spark execution plan?**
    *   The sorting of data within partitions is skipped.
    *   The number of tasks performing the aggregation is reduced.
    *   The **shuffle operation required to group the data is avoided or minimized**.
    *   Data is automatically cached in memory before aggregation.

8.  **According to the information provided, what is the general recommended size range for an optimal bucket in Spark?**
    *   10 to 50 MB.
    *   **128 to 200 MB**.
    *   500 MB to 1 GB.
    *   Greater than 1 GB.

9.  **What is the fundamental formula suggested for estimating the optimal number of buckets for a DataFrame of size `X` with an optimal bucket size `B`?**
    *   `X * B`
    *   `B / X`
    *   **`X / B`**.
    *   `sqrt(X / B)`

10. **What is the term used to describe the optimization where Spark, during a filter operation on a bucketed column, only reads the buckets that could potentially contain the data being filtered?**
    *   Partition elimination.
    *   Predicate pushdown.
    *   **Bucket pruning**.
    *   Filter indexing.

11. **When examining the Spark physical plan after joining two non-bucketed DataFrames, what operator(s) related to data redistribution are most likely to be observed that would ideally be absent when joining correctly bucketed DataFrames?**
    *   `Sort` and `MergeJoin`.
    *   `Collect` and `Take`.
    *   **`Exchange` (for shuffle)**.
    *   `Filter` and `Project`.

12. **Estimating the size of a DataFrame to determine the optimal number of buckets requires considering which of the following factors according to the provided formula?**
    *   Only the number of records.
    *   The number of records and the number of partitions.
    *   **The number of records, the number of variables (columns), and the approximate size of each column**.
    *   The storage format of the DataFrame and the compression codec used.